# Silver: Join weather and electricity prices
Performs a LEFT JOIN of electricity prices against Sola weather observations on the UTC hour, tags match status without imputation, and writes to the joined Silver Delta table.

In [0]:
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = spark.sql("SELECT current_catalog()").first()[0]

dbutils.widgets.text("date_from", "2025-01-01")
dbutils.widgets.text("date_to", "2025-01-07")
dbutils.widgets.text("run_id", "")

DATE_FROM = dbutils.widgets.get("date_from")
DATE_TO = dbutils.widgets.get("date_to")
RUN_ID = dbutils.widgets.get("run_id") or str(uuid.uuid4())

In [0]:
# Load cleaned silver sources for the selected date range
price_df = (
    spark.table(f"{CATALOG}.silver.electricity_price_hourly")
    .filter(F.to_date("time_start_local").between(DATE_FROM, DATE_TO))
)

weather_df = (
    spark.table(f"{CATALOG}.silver.weather_hourly")
    .filter(F.col("station_id") == "SN44560")
)

In [0]:
# Perform LEFT JOIN on UTC timestamp and evaluate match status
joined_df = (
    price_df.alias("price")
    .join(
        weather_df.alias("weather"),
        F.col("price.time_start_utc") == F.col("weather.time_start_utc"),
        "left"
    )
    .select(
        F.col("price.price_area"),
        F.lit("SN44560").alias("station_id"),
        F.col("price.time_start_utc"),
        F.col("price.time_end_utc"),
        F.col("price.time_start_local"),
        F.col("price.nok_per_kwh"),
        F.col("price.eur_per_kwh"),
        F.col("price.exchange_rate"),
        F.col("weather.air_temperature_c"),
        F.col("weather.wind_speed_mps"),
        F.col("weather.precipitation_1h_mm")
    )
    .withColumn(
        "weather_match_status",
        F.when(
            F.col("air_temperature_c").isNull()
            & F.col("wind_speed_mps").isNull()
            & F.col("precipitation_1h_mm").isNull(),
            F.lit("missing")
        )
        .when(
            F.col("air_temperature_c").isNull()
            | F.col("wind_speed_mps").isNull(),
            F.lit("partial")
        )
        .otherwise(F.lit("matched"))
    )
    .withColumn("processed_at_utc", F.current_timestamp())
)

In [0]:
# Validate uniqueness and required fields before writing
duplicate_keys = (
    joined_df.groupBy("price_area", "station_id", "time_start_utc")
    .count().filter("count > 1").count()
)

required_errors = joined_df.filter(
    F.col("time_start_utc").isNull()
    | F.col("nok_per_kwh").isNull()
).count()

if duplicate_keys or required_errors or joined_df.count() == 0:
    raise ValueError(
        "Joined Silver is empty or contains duplicate/null required keys"
    )

# Upsert MERGE into joined Silver table
(
    DeltaTable.forName(
        spark, f"{CATALOG}.silver.weather_price_hourly"
    ).alias("target")
    .merge(
        joined_df.alias("source"),
        """
        target.price_area = source.price_area
        AND target.station_id = source.station_id
        AND target.time_start_utc = source.time_start_utc
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
# Calculate match rate and log data quality result
total_rows = joined_df.count()
matched_rows = joined_df.filter(
    F.col("weather_match_status") == "matched"
).count()
match_rate = matched_rows / total_rows if total_rows else 0.0

checks = [
    (RUN_ID, "silver_join", "joined_keys_unique", "ERROR",
     duplicate_keys == 0, float(duplicate_keys),
     "Joined business keys must be unique",
     datetime.now(timezone.utc).replace(tzinfo=None)),
    (RUN_ID, "silver_join", "weather_match_rate", "WARNING",
     match_rate >= 0.95, float(match_rate),
     f"Weather match rate is {match_rate:.2%}",
     datetime.now(timezone.utc).replace(tzinfo=None))
]

dq_df = spark.createDataFrame(checks, """
    run_id string, layer string, check_name string, severity string,
    passed boolean, actual_value double, message string,
    checked_at_utc timestamp
""")

(
    DeltaTable.forName(
        spark, f"{CATALOG}.ops.data_quality_results"
    ).alias("target")
    .merge(
        dq_df.alias("source"),
        """
        target.run_id = source.run_id
        AND target.layer = source.layer
        AND target.check_name = source.check_name
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(f"Weather match rate: {match_rate:.2%}")
display(joined_df.orderBy("time_start_utc").limit(10))